# Clean the second half of 2025

*Data Analysis in Python · Topic 3.4*

July to December 2025 is going into a board report, and the stacked exports have rows that don't belong in it. These six files already have their dates and revenue in the right types.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell and the cell that stacks the six files into `h2`.
2. In the first cell marked ✏️, create:
   - `duplicates`: the number of rows in `h2` that repeat an earlier row
   - `deduped`: `h2` without them
3. In the second cell marked ✏️, create:
   - `future_rows`: the number of rows in `deduped` dated after `END`
   - `test_rows`: the number of rows in `deduped`, up to `END`, whose `session_medium` is `test`
   - `clean`: `deduped` without the future rows and without the test rows
4. Keep every other row, including the refunds (negative revenue).
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `h2` has 11,923 rows. `duplicates` is `618` and `deduped` has 11,305 rows. `future_rows` is `18` and `test_rows` is `19`. `clean` has 11,268 rows and 819,781 sessions, and still has its 1,416 refund rows.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-07", "2025-08", "2025-09", "2025-10", "2025-11", "2025-12"]
END = "2025-12-31"

## The six files, stacked

In [ ]:
frames = []
for month in MONTHS:
    frames.append(pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv", parse_dates=["date"]))
h2 = pd.concat(frames, ignore_index=True)
len(h2)

## Rows that don't belong

In [ ]:
# ✏️ Create duplicates and deduped


In [ ]:
# ✏️ Create future_rows, test_rows and clean


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    h2 = found.get("h2")
    duplicates = found.get("duplicates")
    deduped = found.get("deduped")
    if not isinstance(h2, pd.DataFrame) or len(h2) != 11923:
        results.append((False, "h2 should be the six files stacked, 11,923 rows. Run the cell that stacks them, and don't change h2 in your cells."))
    elif duplicates == 618 and isinstance(deduped, pd.DataFrame) and len(deduped) == 11305:
        results.append((True, "618 rows were copies from overlapping exports, and deduped has the other 11,305."))
    elif duplicates == 1871 or (isinstance(deduped, pd.DataFrame) and len(deduped) == 10052):
        results.append((False, "1,871 rows share a date, source, medium and campaign with another row, but most of them are real rows with different numbers. Compare whole rows: h2.duplicated(), without subset."))
    elif isinstance(deduped, pd.DataFrame) and len(deduped) == 11923 - 2 * 618:
        results.append((False, "deduped has 10,687 rows: both copies of each duplicate are gone. drop_duplicates() keeps the first copy; duplicated(keep=False) is for looking, not for removing."))
    elif duplicates == 618:
        results.append((False, f"duplicates is right, but deduped should have 11,305 rows. It has {f'{len(deduped):,}' if isinstance(deduped, pd.DataFrame) else repr(deduped)}: use h2.drop_duplicates()."))
    else:
        results.append((False, f"duplicates should be h2.duplicated().sum(), 618. It is {duplicates!r}."))

    future_rows = found.get("future_rows")
    test_rows = found.get("test_rows")
    if future_rows == 18:
        results.append((True, "18 rows are dated after December 31, 2025: an export bug, not traffic."))
    else:
        results.append((False, f"future_rows should be the number of rows in deduped dated after END: (deduped[\"date\"] > END).sum(). It is {future_rows!r}."))
    if test_rows == 19:
        results.append((True, "19 rows are the store's own test traffic (medium test, campaign internal_test)."))
    elif test_rows == 20:
        results.append((False, "test_rows is 20: one test row is a copy from an overlapping export. Count in deduped, after removing duplicates."))
    else:
        results.append((False, f"test_rows should be the number of rows in deduped, up to END, whose session_medium is test. It is {test_rows!r}."))

    clean = found.get("clean")
    if not isinstance(clean, pd.DataFrame):
        results.append((False, "clean isn't a DataFrame yet. Keep deduped's rows up to END whose session_medium isn't test."))
    elif len(clean) == 11268 and clean["sessions"].sum() == 819781 and (clean["revenue"] < 0).sum() == 1416:
        results.append((True, "clean has 11,268 rows and 819,781 sessions, refunds included."))
    elif len(clean) == 9852:
        results.append((False, "clean has 9,852 rows: the 1,416 refund rows are gone. Refunds are real money going back. Keep the rows with negative revenue."))
    elif len(clean) == 11286:
        results.append((False, "clean has 11,286 rows: the 18 future rows are still in. Keep the rows with date <= END."))
    elif len(clean) == 11287:
        results.append((False, "clean has 11,287 rows: the 19 test rows are still in. Keep the rows whose session_medium isn't test."))
    elif len(clean) == 11885:
        results.append((False, "clean has 11,885 rows: the duplicates are still in. Start from deduped, not h2."))
    else:
        results.append((False, f"clean has {len(clean):,} rows, but it should have 11,268: deduped without the future rows and the test rows."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
duplicates = h2.duplicated().sum()
deduped = h2.drop_duplicates()
duplicates, len(deduped)
```

```python
future_rows = (deduped["date"] > END).sum()
in_range = deduped[deduped["date"] <= END]
test_rows = (in_range["session_medium"] == "test").sum()
clean = in_range[in_range["session_medium"] != "test"]
future_rows, test_rows, len(clean), clean["sessions"].sum()
```

</details>